# Feature Selection

In [114]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

np.random.seed(101)

In [115]:
os.chdir("/Users/julianguyen/Documents/multimodal-pmlb")
print(f"Current directory: {os.getcwd()}")

Current directory: /Users/julianguyen/Documents/multimodal-pmlb


### Load in matrices

In [116]:
atc = pd.read_csv("data/procdata/files/atac.csv", index_col=0).T
rna = pd.read_csv("data/procdata/files/rna.csv", index_col=0).T
cnv = pd.read_csv("data/procdata/files/cnv.csv", index_col=0).T
mut = pd.read_csv("data/procdata/files/mut.csv", index_col=0).T

# load in targets
targets = pd.read_csv("data/procdata/files/meta.csv").set_index("PMLB_organoidID").loc[:, ["doubling_rate", "primary_tumor_site"]]

### Create train test splits

In [117]:
assert atc.index.equals(rna.index)
assert atc.index.equals(cnv.index)
assert atc.index.equals(mut.index)
assert atc.index.equals(targets.index)

atc_train, atc_test, rna_train, rna_test, cnv_train, cnv_test, mut_train, mut_test, y_train, y_test = train_test_split(
    atc, rna, cnv, mut, targets,
    test_size=0.2,
    random_state=101
)
dt_train = y_train["doubling_rate"]

### Identify features correlated with doubling rate

In [118]:
def important_features(train, dt, thres):

  # initialize dictionary to hold correlation results
  corr_dict = {}

  # correlate exp of each gene to drug response
  for feature in train.columns:
    corr_dict[feature] = train[feature].corr(dt)
  correlations = pd.DataFrame.from_dict(corr_dict, orient='index', columns=['Correlation'])

  # count number of univariable associations that meet the threshold
  num_pos = (correlations['Correlation'] > thres).sum()
  num_neg = (correlations['Correlation'] < -thres).sum()

  print('Selected threshold:', thres)
  print('Num features with positive correlation > threshold:', num_pos)
  print('Num features with |negative correlation| > threshold:', num_neg)
  print('Total number of features:', str(num_pos + num_neg))

  # identify features that pass selected threshold
  keep = correlations[correlations['Correlation'].abs() > thres].index

  # subset training dataframe to only genes of interest
  X_subset = train[keep]

  return X_subset

In [119]:
atc_train = important_features(atc_train, dt_train, 0.3)

Selected threshold: 0.3
Num features with positive correlation > threshold: 3080
Num features with |negative correlation| > threshold: 3865
Total number of features: 6945


In [120]:
rna_train = important_features(rna_train, dt_train, 0.2)

Selected threshold: 0.2
Num features with positive correlation > threshold: 775
Num features with |negative correlation| > threshold: 5184
Total number of features: 5959


In [121]:
cnv_train = important_features(cnv_train, dt_train, 0.15)

Selected threshold: 0.15
Num features with positive correlation > threshold: 3317
Num features with |negative correlation| > threshold: 2318
Total number of features: 5635


In [122]:
mut_train = important_features(mut_train, dt_train, 0.1)

Selected threshold: 0.1
Num features with positive correlation > threshold: 1998
Num features with |negative correlation| > threshold: 2988
Total number of features: 4986


### Identify correlated features

In [123]:
def corr_features(X_subset, thres):

    # correlate exp of remaining genes
    corr_mat = X_subset.corr(method='pearson', min_periods=1)
    vals = corr_mat.values.copy()
    np.fill_diagonal(vals, 0)
    corr_mat.iloc[:, :] = vals  

    corr_pairs = (corr_mat.abs() > thres)
    correlated = set() # initialize set to store correlated features

    # loop through correlated pairs
    for i in range(corr_pairs.shape[0]):
        for j in range(i + 1, corr_pairs.shape[1]):

            # if True (highly correlated)
            if corr_pairs.iloc[i, j]:

                #print(corr_mat.columns[i], corr_mat.columns[j])

                # add one of the genes to the set
                correlated.add(corr_mat.columns[i])

    #print('Highly correlated features:', correlated)
    print('Num correlated features:', len(correlated))

    print('Original number of feaures:', X_subset.shape[1])

    # remove correlated genes
    X_subset = X_subset.drop(columns=list(correlated))
    print('Number of features remaining:', X_subset.shape[1])

    return X_subset

In [124]:
atc_train = corr_features(atc_train, 0.8)

Num correlated features: 1188
Original number of feaures: 6945
Number of features remaining: 5757


In [125]:
rna_train = corr_features(rna_train, 0.8)

Num correlated features: 2734
Original number of feaures: 5959
Number of features remaining: 3225


In [126]:
cnv_train = corr_features(cnv_train, 0.8)

Num correlated features: 5574
Original number of feaures: 5635
Number of features remaining: 61


In [127]:
mut_train = corr_features(mut_train, 0.8)

Num correlated features: 4329
Original number of feaures: 4986
Number of features remaining: 657


### Match features in test (validation) datasets

In [128]:
atc_test = atc_test.loc[:, atc_train.columns]
rna_test = rna_test.loc[:, rna_train.columns]
cnv_test = cnv_test.loc[:, cnv_train.columns]
mut_test = mut_test.loc[:, mut_train.columns]

### Save new files

In [129]:
# save train files
atc_train.to_csv("data/procdata/model_inputs/atc_train.csv", index=True)
rna_train.to_csv("data/procdata/model_inputs/rna_train.csv", index=True)
cnv_train.to_csv("data/procdata/model_inputs/cnv_train.csv", index=True)
mut_train.to_csv("data/procdata/model_inputs/mut_train.csv", index=True)
y_train.to_csv("data/procdata/model_inputs/meta_train.csv", index=True)

# save test files
atc_test.to_csv("data/procdata/model_inputs/atc_test.csv", index=True)
rna_test.to_csv("data/procdata/model_inputs/rna_test.csv", index=True)
cnv_test.to_csv("data/procdata/model_inputs/cnv_test.csv", index=True)
mut_test.to_csv("data/procdata/model_inputs/mut_test.csv", index=True)
y_test.to_csv("data/procdata/model_inputs/meta_test.csv", index=True)
